# 02. Modelo Predictivo - El consumidor disputara la queja?


## Librerias

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, mean_absolute_error
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_recall_curve










## Carga del dataset limpio

In [5]:
dt_quejas = pd.read_csv('..\data\dt_quejas_corr.csv')
dt_quejas.head(20)


,Complaint ID,Product,Sub-product,Issue,State,ZIP code,Date received,Date sent to company,Company,Company response,Timely response?,Consumer disputed?,Company_grouped_filtered,Difference in days
0,1291006,Debt collection,Credit card,Communication tactics,TX,76119,2015-03-19,2015-03-19,"Premium Asset Services, LLC",In progress,Yes,Pending,Low Count Companies,0
1,1290580,Debt collection,Medical,Cont'd attempts collect debt not owed,TX,77479,2015-03-19,2015-03-19,Accounts Receivable Consultants Inc.,Closed with explanation,Yes,No,Low Count Companies,0
2,1290564,Mortgage,FHA mortgage,"Application, originator, mortgage broker",MA,02127,2015-03-19,2015-03-19,RBS Citizens,Closed with explanation,Yes,Yes,RBS Citizens,0
3,1291615,Credit card,Unknown,Other,CA,92592,2015-03-19,2015-03-19,Navy FCU,In progress,Yes,Pending,Navy FCU,0
4,1292165,Debt collection,Non-federal student loan,Cont'd attempts collect debt not owed,OH,43068,2015-03-19,2015-03-19,Transworld Systems Inc.,In progress,Yes,Pending,Transworld Systems Inc.,0
5,1291176,Debt collection,Payday loan,Communication tactics,OH,43068,2015-03-19,2015-03-19,ACE Cash Express Inc.,In progress,Yes,Pending,ACE Cash Express Inc.,0
6,1288848,Consumer loan,Installment loan,Managing the loan or lease,OH,44241,2015-03-18,2015-03-18,"CashCall, Inc.",Closed with explanation,Yes,Yes,"CashCall, Inc.",0
7,1288788,Debt collection,Payday loan,Communication tactics,CA,95124,2015-03-18,2015-03-18,ACE Cash Express Inc.,Closed with explanation,Yes,No,ACE Cash Express Inc.,0
8,1288324,Debt collection,"Other (phone, health club, etc.)",Cont'd attempts collect debt not owed,NJ,07067,2015-03-18,2015-03-18,"Credit Protection Association, L.P.",Closed with non-monetary relief,Yes,No,Low Count Companies,0
9,1288304,Debt collection,Payday loan,Taking/threatening an illegal action,TX,77433,2015-03-18,2015-03-18,Cottonwood Financial Ltd.,Closed with explanation,Yes,Yes,Low Count Companies,0


## Modelos a realizar

#### Definimos X e Y

In [6]:
y = dt_quejas['Consumer disputed?']

caracteristicas = [
    'Product',
    'Issue',
    'Company response',
    'Timely response?',
    'Company_grouped_filtered',
    'State',
    'Difference in days'
]

caracteristicas_con_subproduct = caracteristicas + ['Sub-product']
caracteristicas_sin_subproduct = caracteristicas.copy()

X_con_subproduct = dt_quejas[caracteristicas_con_subproduct]
X_sin_subproduct = dt_quejas[caracteristicas_sin_subproduct]

#### Modelo con Random Forest

In [9]:
def entrenar_modelo(X, y, labels):
    categoricas = X.select_dtypes(include='object').columns.tolist()
    numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
        ],
        remainder='passthrough'
    )

    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: sólo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return clf


In [10]:
modelo_con_sub = entrenar_modelo(X_con_subproduct, y, labels=['No', 'Pending', 'Yes'])
modelo_sin_subproduct = entrenar_modelo(X_sin_subproduct, y,labels=['No', 'Pending', 'Yes'])


 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.85      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.19       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.78      0.76      5632

 Confusion Matrix:
 [[3625    0  457]
 [   0  608    0]
 [ 797    0  145]]

 Métricas adicionales:
Accuracy: 0.7773
Precision (macro): 0.6869
Recall (macro): 0.6807
F1-score (macro): 0.6801
F1-score (weighted): 0.7573
ROC AUC (OvR): 0.7880
 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.86      0.84      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.19      0.21       942

    accuracy                           0.76      5632
   macro avg       0.69      0.68      0.68      5632
wei

El modelo de Random Forest que incluye a sub-product, obtuvo un accuracy del 77.7%, acompañado de un F1-score ponderado de 0.7573. En cuanto al F1-score macro, que otorga igual importancia a cada clase, se situó en 0.6801, lo que indica un rendimiento aceptable considerando el desbalance entre clases. La clase "No" (que el consumidor no disputó la queja) fue correctamente identificada con una precisión del 82% y una sensibilidad del 89%, mientras que la clase "Pending" (proceso aún en curso) fue perfectamente clasificada. Sin embargo, el desempeño para la clase "Yes" (quejas que fueron disputadas por el consumidor) fue considerablemente menor, con un F1-score de solo 0.19, reflejando una dificultad clara del modelo para identificar estos casos. El ROC AUC multiclase alcanzó un valor de 0.788, lo que muestra una buena capacidad del modelo para discriminar entre clases incluso en contextos multiclase complejos.

En este segundo modelo se eliminó la variable Sub-product, y los resultados se mantuvieron en una línea similar aunque ligeramente inferiores. El accuracy bajó a 76.1% y el F1-score ponderado a 0.7513, lo que sugiere una leve pérdida de rendimiento general. De nuevo, las clases "No" y "Pending" fueron clasificadas correctamente, con métricas prácticamente idénticas. En la clase "Yes", el modelo logró un F1-score algo superior (0.21), pero aún muy bajo para ser considerado útil de forma aislada. El ROC AUC descendió un poco hasta 0.7847.

Aambos modelos enfrentan una limitación común importante: la incapacidad para predecir correctamente las disputas ("Yes"), probablemente debido al desbalance de clases y la posible falta de señales predictivas claras para este comportamiento.A pesar de ello, las métricas globales como el ROC AUC y el F1 ponderado indican que el modelo con Sub-product es ligeramente superior y, por tanto, sería el más recomendable de los dos para continuar con el proceso de desarrollo o despliegue.

##### Oversampling de la clase mayoritaria

In [12]:
def entrenar_modelo_con_smote(X, y, labels):

    categoricas = X.select_dtypes(include='object').columns.tolist()
    numericas= X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
        ],
        remainder='passthrough'
    )

    X_train_enc = preprocessor.fit_transform(X_train)
    X_test_enc = preprocessor.transform(X_test)

    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train_enc, y_train)

    clf = RandomForestClassifier(class_weight='balanced', random_state=42)
    clf.fit(X_resampled, y_resampled)

    y_pred = clf.predict(X_test_enc)
    y_proba = clf.predict_proba(X_test_enc)

    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: sólo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return clf

In [13]:
modelo_con_sub = entrenar_modelo_con_smote(X_con_subproduct, y, labels=['No', 'Pending', 'Yes'])
modelo_sin_subproduct = entrenar_modelo_con_smote(X_sin_subproduct, y,labels=['No', 'Pending', 'Yes'])

Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.91      0.86      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.25      0.13      0.17       942

    accuracy                           0.79      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.79      0.76      5632

Confusion Matrix:
 [[3723    0  359]
 [   0  608    0]
 [ 821    0  121]]

 Métricas adicionales:
Accuracy: 0.7905
Precision (macro): 0.6905
Recall (macro): 0.6802
F1-score (macro): 0.6778
F1-score (weighted): 0.7621
ROC AUC (OvR): 0.7886
Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.86      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.18       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68      5632
weight

Se han entrenado dos modelos de clasificación para predecir si un consumidor disputará una queja, esta vez incluyendo un enfoque de balanceo de clases mediante SMOTE (Synthetic Minority Oversampling Technique). El objetivo era mejorar la capacidad del modelo para detectar correctamente la clase minoritaria: "Yes". Aunque el uso de SMOTE ha permitido una ligera mejora en métricas generales como el accuracy, pero no ha logrado mejorar de forma significativa el rendimiento sobre la clase minoritaria "Yes". La precisión y el F1-score se mantuvieron prácticamente iguales, y el recall incluso disminuyó ligeramente.

##### Hiperparametrizacion con y sin SMOTE

In [16]:
def entrenar_modelo_gs(X, y, labels):
    categoricas = X.select_dtypes(include='object').columns.tolist()
    numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
        ],
        remainder='passthrough'
    )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    param_grid = {
        'classifier__n_estimators': [100],
        'classifier__max_depth': [None, 10],
        'classifier__min_samples_split': [2],
        'classifier__min_samples_leaf': [1],
        'classifier__class_weight': ['balanced']
    }

    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: solo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    print(f"\n Mejores parámetros encontrados: {grid_search.best_params_}")

    return best_model


In [18]:
modelo_gs = entrenar_modelo_gs(X_con_subproduct, y, labels=['No', 'Pending', 'Yes'])

Fitting 3 folds for each of 2 candidates, totalling 6 fits
 Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.89      0.85      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.24      0.15      0.19       942

    accuracy                           0.78      5632
   macro avg       0.69      0.68      0.68      5632
weighted avg       0.74      0.78      0.76      5632

 Confusion Matrix:
 [[3625    0  457]
 [   0  608    0]
 [ 797    0  145]]

 Métricas adicionales:
Accuracy: 0.7773
Precision (macro): 0.6869
Recall (macro): 0.6807
F1-score (macro): 0.6801
F1-score (weighted): 0.7573
ROC AUC (OvR): 0.7880

 Mejores parámetros encontrados: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}


Tras hiperparametrizar sin SMOTE, el rendimiento global fue correcto: accuracy 0.777, F1 macro 0.680 y ROC AUC (OvR) 0.788. La clase Pending se clasifica de forma perfecta (F1=1.00), la clase No presenta buen equilibrio (precision=0.82, recall=0.89, F1=0.85) y la clase Yes es claramente la más problemática (precision=0.24, recall=0.15, F1=0.19). La matriz de confusión confirma que muchos “Yes” reales se confunden con “No”. En resumen, este baseline demuestra buen desempeño agregado, pero insuficiente capacidad para detectar disputas (“Yes”).

In [14]:
def entrenar_modelo_gs(X, y, labels, use_smote=True):

    categoricas = X.select_dtypes(include='object').columns.tolist()
    numericas = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
        ],
        remainder='passthrough'
    )

    if use_smote:
        pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=42)),
            ('classifier', RandomForestClassifier(random_state=42))
        ])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10],
            'classifier__min_samples_split': [2],
            'classifier__min_samples_leaf': [1],
            'classifier__class_weight': ['balanced'] }
    else:
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(random_state=42))
        ])
        param_grid = {
            'classifier__n_estimators': [100],
            'classifier__max_depth': [None, 10],
            'classifier__min_samples_split': [2],
            'classifier__min_samples_leaf': [1],
            'classifier__class_weight': ['balanced']
        }

    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test_bin = lb.transform(y_test)

    if y_test_bin.shape[1] == 1:
        print("ROC AUC: solo hay una clase activa, no se puede calcular.")
    else:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    print(f"\n Mejores parámetros encontrados: {grid_search.best_params_}")
    print(f" SMOTE activado: {use_smote}")

    return best_model

In [15]:
labels = ['No', 'Pending', 'Yes']

modelo_smote = entrenar_modelo_gs(X_con_subproduct, y, labels, use_smote=True)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
 Classification Report:
               precision    recall  f1-score   support

          No       0.84      0.60      0.70      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.23      0.51      0.32       942

    accuracy                           0.63      5632
   macro avg       0.69      0.70      0.67      5632
weighted avg       0.76      0.63      0.67      5632

 Confusion Matrix:
 [[2441    0 1641]
 [   0  608    0]
 [ 457    0  485]]

 Métricas adicionales:
Accuracy: 0.6275
Precision (macro): 0.6901
Recall (macro): 0.7043
F1-score (macro): 0.6719
F1-score (weighted): 0.6678
ROC AUC (OvR): 0.7947

 Mejores parámetros encontrados: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
 SMOTE activado: True


En esta segunda iteración repetí el mismo pipeline con Random Forest, pero integrando SMOTE antes del clasificador para balancear las clases en el conjunto de entrenamiento. El efecto principal es el esperado en escenarios desbalanceados: la clase Yes mejora su recall del 0.15 al 0.51 (F1=0.32), a costa de una caída del accuracy al 0.63 y de un descenso notable del recall en No (0.60). La clase Pending sigue separándose de forma perfecta (F1=1.00). La matriz de confusión muestra que ahora el modelo “arriesga” más con Yes (menos falsos negativos) pero aumenta los falsos positivos sobre No. A nivel agregado, F1 macro sube ligeramente (0.672) y ROC AUC se mantiene sólido (0.795), lo que sugiere que el modelo discrimina bien, aunque el punto de operación (frontera de decisión) favorece la sensibilidad de Yes. En síntesis, SMOTE cumple el objetivo de detectar mejor la clase minoritaria, penalizando el rendimiento global: una decisión coherente si el objetivo de negocio es no perder disputas aunque haya que revisar más casos.

##### Modelo XGBoost

In [41]:
def entrenar_xgb_basico(X, y, labels, usar_pesos_clase=True):
    
    label_to_num = {label: i for i, label in enumerate(labels)}
    num_to_label = {i: label for label, i in label_to_num.items()}
    y_num = y.map(label_to_num)

    
    cat_features = X.select_dtypes(include='object').columns.tolist()

  
    X_train, X_test, y_train_num, y_test_num = train_test_split(
        X, y_num, stratify=y_num, test_size=0.2, random_state=42
    )


    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)],
        remainder='passthrough'
    )


    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=len(labels),
        eval_metric='mlogloss',
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42
    )

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb)
    ])

    fit_kwargs = {}
    if usar_pesos_clase:
        classes_num = np.arange(len(labels))
        pesos = compute_class_weight(
            class_weight='balanced',
            classes=classes_num,
            y=y_train_num.values
        )
        mapa_pesos = {cls: w for cls, w in zip(classes_num, pesos)}
        sample_weight = pd.Series(y_train_num).map(mapa_pesos).values
        fit_kwargs['classifier__sample_weight'] = sample_weight


    pipe.fit(X_train, y_train_num, **fit_kwargs)


    y_pred_num  = pipe.predict(X_test)
    y_proba     = pipe.predict_proba(X_test)


    y_test = pd.Series(y_test_num).map(num_to_label)
    y_pred = pd.Series(y_pred_num).map(num_to_label)

    print("Classification Report:\n",
          classification_report(y_test_num, y_pred_num,
                                labels=list(range(len(labels))),
                                target_names=labels))
    print("Confusion Matrix:\n",
          confusion_matrix(y_test_num, y_pred_num, labels=list(range(len(labels)))))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test_num, y_pred_num):.4f}")
    print(f"Precision (macro): {precision_score(y_test_num, y_pred_num, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test_num, y_pred_num, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test_num, y_pred_num, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test_num, y_pred_num, average='weighted'):.4f}")


    lb = LabelBinarizer().fit(y_test_num)
    y_test_bin = lb.transform(y_test_num)
    if y_test_bin.shape[1] > 1:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return pipe

In [42]:
labels = ['No', 'Pending', 'Yes']

features_with_subproduct = [
    'Product', 'Sub-product', 'Issue', 'State', 'ZIP code',
    'Company_grouped_filtered', 'Company response', 'Timely response?',
    'Difference in days'
]
X_with_subproduct = dt_quejas[features_with_subproduct]
y = dt_quejas['Consumer disputed?']

modelo_xgb_con_sub = entrenar_xgb_basico(
    X=X_with_subproduct,
    y=y,
    labels=labels,
    usar_pesos_clase=False
)


Classification Report:
               precision    recall  f1-score   support

          No       0.82      0.99      0.90      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.62      0.04      0.08       942

    accuracy                           0.84      5632
   macro avg       0.81      0.68      0.66      5632
weighted avg       0.80      0.84      0.77      5632

Confusion Matrix:
 [[4057    0   25]
 [   0  608    0]
 [ 901    0   41]]

 Métricas adicionales:
Accuracy: 0.8356
Precision (macro): 0.8132
Recall (macro): 0.6791
F1-score (macro): 0.6596
F1-score (weighted): 0.7721
ROC AUC (OvR): 0.8095


En esta tercera iteración cambié a XGBoost dentro de un pipeline idéntico, para luego entrenar un clasificador multiclase. Para luego mitigar el desbalance, aplicando pesos calculados.  n test, el modelo logra accuracy alto (0.836), ROC AUC macro 0.810 y un comportamiento excelente en Pending (F1=1.00) y sólido en No (F1=0.90), pero fracasa en “Yes”: aunque su precision es 0.62, el recall cae a 0.04 (F1=0.08), es decir, casi no captura disputas reales. La matriz de confusión confirma que la mayoría de “Yes” se están etiquetando como “No”. En síntesis, XGBoost con esta configuración separa bien a nivel de probabilidad (AUC alto) y maximiza acierto global, pero no sirve para el objetivo de negocio (detectar “Yes”); por eso, en los siguientes pasos ajustare pesos específicos a “Yes” y umbrales de decisión para elevar su recall, asumiendo la penalización en accuracy que conlleva.

In [ ]:
def entrenar_xgb_pesos(X, y, labels):
    cat_features = X.select_dtypes(include='object').columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)],
        remainder='passthrough'
    )

    counts = Counter(y_train)
    total = sum(counts.values())
    pesos = {cls: total / count for cls, count in counts.items()}
    sample_weight = y_train.map(pesos).values

    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=len(labels),
        eval_metric='mlogloss',
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42
    )

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb)
    ])

    pipe.fit(X_train, y_train, classifier__sample_weight=sample_weight)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    print(" Classification Report:\n", classification_report(y_test, y_pred))
    print(" Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

    lb = LabelBinarizer().fit(y_test)
    y_test_bin = lb.transform(y_test)
    if y_test_bin.shape[1] > 1:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

    return pipe


In [ ]:
labels = ['No', 'Pending', 'Yes']

features_with_subproduct = [
    'Product', 'Sub-product', 'Issue', 'State', 'ZIP code',
    'Company_grouped_filtered', 'Company response', 'Timely response?',
    'Difference in days'
]

X = dt_quejas[features_with_subproduct]
y = dt_quejas['Consumer disputed?']

label_to_num = {label: i for i, label in enumerate(labels)} 
y_num = y.map(label_to_num)

modelo_xgb_pesos = entrenar_xgb_pesos(X=X, y=y_num, labels=labels)

 Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.60      0.71      4082
           1       1.00      1.00      1.00       608
           2       0.24      0.54      0.33       942

    accuracy                           0.64      5632
   macro avg       0.70      0.71      0.68      5632
weighted avg       0.76      0.64      0.68      5632

 Confusion Matrix:
 [[2464    0 1618]
 [   0  608    0]
 [ 433    0  509]]

Accuracy: 0.6358
F1 Macro: 0.6793
ROC AUC (OvR): 0.8100


Aqui, etrené XGBoost con ponderación explícita por clase: calculé pesos inversos a la frecuencia en el train y los pasé como sample_weight dentro del pipeline (tras el OneHotEncoder). De tal forma que desplazamos la frontera de decisión hacia “Yes”, que era nuestro objetivo. El resultado refleja que el recall de Yes sube con fuerza hasta 0.54 (F1=0.33), mientras que el recall de No cae a 0.60, y la accuracy global baja a 0.64. La clase Pending se mantiene perfecta (F1=1.00). A nivel agregado, el F1 macro mejora respecto al baseline centrado en accuracy y el ROC AUC se mantiene alto (0.81), lo que sugiere que el modelo sigue discriminando bien, pero ahora prioriza sensibilidad para Yes. En términos de negocio, este enfoque tiene sentido si preferimos no perder disputas aunque implique revisar más falsos positivos.

In [ ]:
def _eval_and_report(model, X_test, y_test, labels):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    print("Classification Report:\n",
          classification_report(y_test, y_pred, labels=list(range(len(labels))), target_names=labels))
    print("Confusion Matrix:\n",
          confusion_matrix(y_test, y_pred, labels=list(range(len(labels)))))

    print("\n Métricas adicionales:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

    lb = LabelBinarizer().fit(y_test)
    y_test_bin = lb.transform(y_test)
    if y_test_bin.shape[1] > 1:
        auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        print(f"ROC AUC (OvR): {auc:.4f}")

In [ ]:
labels = ['No', 'Pending', 'Yes']

features = [
    'Product', 'Sub-product', 'Issue', 'State', 'ZIP code',
    'Company_grouped_filtered', 'Company response', 'Timely response?',
    'Difference in days'
]

X = dt_quejas[features]
y = dt_quejas['Consumer disputed?']

def entrenar_xgb_grid_macroF1(X, y, labels, yes_weight_factor=None):

    label_to_num = {label: i for i, label in enumerate(labels)}
    y_num = y.map(label_to_num)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_num, stratify=y_num, test_size=0.2, random_state=42
    )
    cat_features = X.select_dtypes(include='object').columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)],
        remainder='passthrough'
    )

    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=len(labels),
        eval_metric='mlogloss',
        random_state=42
    )

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb)
    ])
    param_grid = {
        'classifier__n_estimators': [300, 600],
        'classifier__max_depth': [4, 6],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__min_child_weight': [1, 5],
        'classifier__subsample': [0.8],
        'classifier__colsample_bytree': [0.8]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    fit_kwargs = {}
    if yes_weight_factor is not None:
        
        yes_id = label_to_num['Yes']
        weights = np.ones_like(y_train, dtype=float)
        weights[y_train == yes_id] *= float(yes_weight_factor)
        fit_kwargs['classifier__sample_weight'] = weights

    grid.fit(X_train, y_train, **fit_kwargs)
    best_model = grid.best_estimator_

    print(f"\n✅ Mejores parámetros (F1 macro): {grid.best_params_}")
    _eval_and_report(best_model, X_test, y_test, labels)
    return best_model

In [47]:
best_xgb_macro = entrenar_xgb_grid_macroF1(X, y, labels, yes_weight_factor=3)


Fitting 3 folds for each of 16 candidates, totalling 48 fits

✅ Mejores parámetros (F1 macro): {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 6, 'classifier__min_child_weight': 1, 'classifier__n_estimators': 600, 'classifier__subsample': 0.8}
Classification Report:
               precision    recall  f1-score   support

          No       0.83      0.87      0.85      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.29      0.23      0.26       942

    accuracy                           0.78      5632
   macro avg       0.71      0.70      0.70      5632
weighted avg       0.76      0.78      0.77      5632

Confusion Matrix:
 [[3556    0  526]
 [   0  608    0]
 [ 722    0  220]]

 Métricas adicionales:
Accuracy: 0.7784
Precision (macro): 0.7087
Recall (macro): 0.7016
F1-score (macro): 0.7038
F1-score (weighted): 0.7681
ROC AUC (OvR): 0.8096


En esta cuarta iteración entrené un XGBoost integrado en el mismo pipeline de preprocesado, pero esta vez hiperparametrizado con GridSearchCV optimizando f1_macro. En test, el rendimiento global mejoró respecto al baseline: accuracy 0.778, F1 macro 0.704, y ROC AUC 0.810. Por clases, Pending sigue perfecto (F1=1.00), No mantiene un buen equilibrio (F1=0.85), y Yes mejora ligeramente frente al Random Forest inicial (F1≈0.26; precision≈0.29, recall≈0.23), aunque continúa siendo la clase más difícil. La matriz de confusión refleja que todavía se confunden muchos “Yes” con “No”, pero menos que antes. En resumen, este XGBoost equilibra mejor el rendimiento entre clases sin sacrificar la capacidad discriminativa general.

In [ ]:
def entrenar_xgb_grid_f1Yes(X, y, labels, yes_weight_factor=None):
    label_to_num = {label: i for i, label in enumerate(labels)}
    yes_id = label_to_num['Yes']

    y_num = y.map(label_to_num)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_num, stratify=y_num, test_size=0.2, random_state=42
    )

    cat_features = X.select_dtypes(include='object').columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)],
        remainder='passthrough'
    )

    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=len(labels),
        eval_metric='mlogloss',
        random_state=42
    )

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', xgb)
    ])

    param_grid = {
        'classifier__n_estimators': [300, 600],
        'classifier__max_depth': [4, 6],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__min_child_weight': [1, 5],
        'classifier__subsample': [0.8],
        'classifier__colsample_bytree': [0.8]
    }

    def f1_yes_only(y_true, y_pred):
        return f1_score(y_true, y_pred, labels=[yes_id], average='macro')

    scorer_yes = make_scorer(f1_yes_only, greater_is_better=True)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=3,
        scoring=scorer_yes,
        n_jobs=-1,
        verbose=1
    )

    fit_kwargs = {}
    if yes_weight_factor is not None:
        weights = np.ones_like(y_train, dtype=float)
        weights[y_train == yes_id] *= float(yes_weight_factor)
        fit_kwargs['classifier__sample_weight'] = weights

    grid.fit(X_train, y_train, **fit_kwargs)
    best_model = grid.best_estimator_

    print(f"\n Mejores parámetros (F1 de 'Yes'): {grid.best_params_}")
    _eval_and_report(best_model, X_test, y_test, labels)
    return best_model

In [49]:
best_xgb_f1yes = entrenar_xgb_grid_f1Yes(X, y, labels, yes_weight_factor=3)


Fitting 3 folds for each of 16 candidates, totalling 48 fits

✅ Mejores parámetros (F1 de 'Yes'): {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 4, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 600, 'classifier__subsample': 0.8}
Classification Report:
               precision    recall  f1-score   support

          No       0.83      0.85      0.84      4082
     Pending       1.00      1.00      1.00       608
         Yes       0.28      0.26      0.27       942

    accuracy                           0.77      5632
   macro avg       0.70      0.70      0.70      5632
weighted avg       0.76      0.77      0.76      5632

Confusion Matrix:
 [[3466    0  616]
 [   0  608    0]
 [ 700    0  242]]

 Métricas adicionales:
Accuracy: 0.7663
Precision (macro): 0.7047
Recall (macro): 0.7020
F1-score (macro): 0.7031
F1-score (weighted): 0.7621
ROC AUC (OvR): 0.8069


Tras reconfigurar un GridSearchCV sobre XGBoost optimizando el F1-score de “Yes” mediante un scorer personalizado, el mejor conjunto resultó en un modelo más conservador que tiende a reducir sobreajuste y controlar falsos positivos al tiempo que concentra capacidad en la clase minoritaria. En test, este modelo ofrece un compromiso razonable: accuracy 0.766, F1 macro 0.703 y ROC AUC 0.807. La clase Yes sube hasta F1≈0.27 (precision≈0.28, recall≈0.26), mejorando frente al estandar obtenido, mientras que No y Pending mantienen buen rendimiento (F1=0.84 y F1=1.00, respectivamente). La matriz de confusión confirma que, aunque aún se confunden “Yes” con “No”, el volumen de aciertos en “Yes” crece sin un colapso del rendimiento global. Teniendo en cuenta que nos interesa ver si el cliente va a reclamar o no, este es el modelo que conservaría como candidato final: equilibra razonablemente el objetivo de detectar disputas con la estabilidad del sistema, y deja margen para afinar el umbral de decisión si se requiere más sensibilidad o más precisión en producción.